# TP 1 — Partie 1 : ingestion en couche Bronze

**Objectif :** charger les données brutes d'Olist dans le Lakehouse, **sans les modifier**.

La couche Bronze est une copie fidèle des sources. On y ajoute seulement deux colonnes techniques :
- `_source` : d'où vient la ligne (fichier ou table) ;
- `_ingere_le` : quand elle a été chargée.

Si une erreur apparaît plus tard dans le pipeline, Bronze permet de tout recalculer sans redemander les données aux systèmes sources.

| Source | Tables | Comment elles arrivent |
|---|---|---|
| Base PostgreSQL (application de vente) | `orders`, `order_items`, `order_payments` | lecture JDBC |
| Fichiers CSV (exports des référentiels) | `customers`, `products`, `sellers`, `category_translation` | dépôt dans `s3://landing/` |

## Étape 0 — Préparer la session

Exécutez la cellule : elle démarre Spark et crée les trois espaces de noms du Lakehouse.

In [ ]:
import os
import datetime
import boto3
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("tp1-bronze").getOrCreate()

for couche in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE NAMESPACE IF NOT EXISTS lakehouse.{couche}")

spark.sql("SHOW NAMESPACES IN lakehouse").show()

Chargez ensuite la base PostgreSQL « source » (lot 1 : état du système au 1er juillet 2018).

In [ ]:
!python /opt/lakehouse/scripts/load_postgres.py --lot 1

## Étape 1 — Déposer les fichiers dans la zone *landing*

Dans une vraie entreprise, ces fichiers seraient déposés automatiquement chaque nuit par un autre système.
Ici, on simule ce dépôt : chaque fichier est copié dans `s3://landing/olist/<table>/date_ingestion=<date>/`.

In [ ]:
s3 = boto3.client("s3", endpoint_url=os.environ["S3_ENDPOINT"])
DATE_INGESTION = datetime.date.today().isoformat()

FICHIERS = {
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

for table, fichier in FICHIERS.items():
    cle = f"olist/{table}/date_ingestion={DATE_INGESTION}/{fichier}"
    s3.upload_file(f"/home/etudiant/work/data/olist/{fichier}", "landing", cle)
    print("déposé :", f"s3://landing/{cle}")

> **❓ Question 1.** Pourquoi range-t-on chaque dépôt dans un dossier `date_ingestion=...` plutôt que d'écraser le fichier précédent ?

## Étape 2 — Le piège du typage automatique

Lisez le fichier des clients en laissant Spark deviner les types (`inferSchema`), puis observez la colonne `customer_zip_code_prefix`.

In [ ]:
devine = (spark.read
          .option("header", True)
          .option("inferSchema", True)
          .csv("s3a://landing/olist/customers/"))
devine.printSchema()
devine.select("customer_zip_code_prefix", "customer_city").show(5)

> **❓ Question 2.** Le préfixe de code postal `09790` (São Bernardo do Campo) devient `9790`. Pourquoi ? Quelle conséquence aurait cette erreur si on l'enregistrait en Bronze ?

## Étape 3 — Ingérer les fichiers CSV en Bronze

Complétez la fonction `bronze_depuis_csv(table)`. Elle doit :
1. lire **en texte** (sans `inferSchema`) tous les fichiers de `s3a://landing/olist/<table>/`, avec la ligne d'en-tête ;
2. ajouter la colonne `_source` (chemin du fichier lu : `F.col("_metadata.file_path")`) et la colonne `_ingere_le` (`F.current_timestamp()`) ;
3. écrire le résultat dans la table Iceberg `lakehouse.bronze.<table>` en la remplaçant si elle existe :
   `df.writeTo("...").using("iceberg").createOrReplace()`.

Le nettoyage des noms de colonnes est fourni : le fichier de traduction commence par un caractère invisible (BOM) qui se retrouverait dans le nom de la première colonne.

In [ ]:
def bronze_depuis_csv(table):
    df = ...  # À COMPLÉTER : lecture des CSV en texte

    # Nettoyage technique des noms de colonnes (caractère BOM éventuel)
    df = df.toDF(*[c.replace("\ufeff", "") for c in df.columns])

    df = ...  # À COMPLÉTER : ajout de _source et _ingere_le

    # À COMPLÉTER : écriture dans lakehouse.bronze.<table>

    return spark.table(f"lakehouse.bronze.{table}").count()


for table in FICHIERS:
    print(f"bronze.{table:22s}", bronze_depuis_csv(table), "lignes")

## Étape 4 — Ingérer la base PostgreSQL en Bronze

Les paramètres de connexion sont fournis. Complétez `bronze_depuis_postgres(table)` sur le même modèle :
lecture JDBC de `ventes.<table>` (`spark.read.format("jdbc").options(**JDBC).option("dbtable", ...)`),
ajout de `_source` (texte fixe, par exemple `"postgres:ventes.orders"`) et `_ingere_le`, écriture Iceberg.

In [ ]:
JDBC = {
    "url": f"jdbc:postgresql://{os.environ['PG_HOST']}:5432/{os.environ['PG_DB']}",
    "user": os.environ["PG_USER"],
    "password": os.environ["PG_PASSWORD"],
    "driver": "org.postgresql.Driver",
}


def bronze_depuis_postgres(table):
    # À COMPLÉTER
    ...

    return spark.table(f"lakehouse.bronze.{table}").count()


for table in ["orders", "order_items", "order_payments"]:
    print(f"bronze.{table:22s}", bronze_depuis_postgres(table), "lignes")

## Étape 5 — Contrôler la couche Bronze

Vérifiez que les sept tables existent et comparez les types : les tables venant de PostgreSQL sont déjà typées, celles venant des CSV sont entièrement en texte.

In [ ]:
spark.sql("SHOW TABLES IN lakehouse.bronze").show()
spark.table("lakehouse.bronze.orders").printSchema()
spark.table("lakehouse.bronze.products").printSchema()

Regardez enfin ce qu'Iceberg a réellement écrit dans le stockage pour une table : des fichiers de données Parquet et des fichiers de métadonnées.
Vous pouvez aussi parcourir le bucket `lakehouse` dans la console RustFS (http://localhost:9001).

In [ ]:
reponse = s3.list_objects_v2(Bucket="lakehouse", Prefix="warehouse/bronze/sellers")
for objet in reponse.get("Contents", []):
    print(f"{objet['Size']:>9} o  {objet['Key']}")

> **❓ Question 3.** La table `bronze.orders` contient environ 86 600 commandes, alors que le jeu Olist en compte près de 100 000. Où sont les autres ? (Indice : relisez la description du lot 1.)

> **❓ Question 4.** Pourquoi ne corrige-t-on pas les données (doublons, valeurs manquantes) dès la couche Bronze ?